# BCE-SASRec: proposed model for MOOCCubeX

**BCE-SASRec** means **Behaviour and Concept-Explainable Self-Attentive Sequential Recommender**. It uses one causal Transformer. Every watched-video token combines video, concept, course, metadata, behaviour, time-gap, and position information.

The notebook implements the complete research pipeline:

- chronological train, validation, and test inputs;
- train-only feature normalization;
- leakage checks;
- multi-task learning for next-video ranking, next-concept alignment, and completion prediction;
- a maximum of 25 epochs with validation-based early stopping;
- best-checkpoint restoration before testing;
- Accuracy@1, Precision, Recall, F1, NDCG, MAP, MRR, mean rank, median rank, catalog coverage, and novelty;
- counterfactual history explanations and concept evidence;
- checkpoints, epoch logs, test results, recommendations, and explanation JSON files saved to Drive.

Expected input directory: `/content/drive/MyDrive/DataCon/processed`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install pyarrow pandas tqdm matplotlib

## 1. Imports and reproducible parameters

The maximum of 25 epochs matches the baseline experiment. Early stopping begins only after 10 epochs and stops after 5 consecutive epochs without a meaningful validation NDCG@10 improvement.

In [ ]:
from pathlib import Path
from dataclasses import dataclass, asdict
import copy, gc, json, math, os, random, time
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

@dataclass
class Config:
    root: str = '/content/drive/MyDrive/DataCon'
    seed: int = 42
    max_len: int = 50
    max_concepts_per_video: int = 12
    hidden_dim: int = 128
    transformer_layers: int = 2
    attention_heads: int = 4
    feedforward_dim: int = 512
    dropout: float = 0.10
    time_buckets: int = 32
    negatives: int = 50
    batch_size: int = 128
    eval_batch_size: int = 128
    max_epochs: int = 25
    minimum_epochs: int = 10
    early_stopping_patience: int = 5
    early_stopping_min_delta: float = 1e-4
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    concept_loss_weight: float = 0.20
    completion_loss_weight: float = 0.10
    gradient_clip: float = 5.0
    use_mixed_precision: bool = False
    ks: tuple = (5, 10, 20)
    num_workers: int = 2

CFG=Config()
ROOT=Path(CFG.root); PROCESSED=ROOT/'processed'; SPLITS=PROCESSED/'splits'; GRAPH=PROCESSED/'graph'
OUT=ROOT/'proposed_bce_sasrec'; CHECKPOINTS=OUT/'checkpoints'; REPORTS=OUT/'reports'; EXPLANATIONS=OUT/'explanations'
for p in [OUT,CHECKPOINTS,REPORTS,EXPLANATIONS]: p.mkdir(parents=True,exist_ok=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

seed_everything(CFG.seed)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device)
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
print(json.dumps(asdict(CFG),indent=2))

## 2. Load train, validation, and test datasets

The input splits were created using per-user chronological leave-two-out. Training data supplies model fitting and normalization. Validation data controls checkpoint selection and early stopping. Test data is used only once after the best checkpoint is restored.

In [ ]:
required=[SPLITS/'train.parquet',SPLITS/'valid.parquet',SPLITS/'test.parquet',
          GRAPH/'video_metadata.parquet',GRAPH/'concept_video_edges.parquet',
          GRAPH/'video_index.parquet',GRAPH/'course_video_edges.parquet']
missing=[str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError(f'Missing preprocessing outputs: {missing}')

train_df=pd.read_parquet(SPLITS/'train.parquet').sort_values(['user_id','timestamp']).reset_index(drop=True)
valid_df=pd.read_parquet(SPLITS/'valid.parquet').sort_values(['user_id','timestamp']).reset_index(drop=True)
test_df=pd.read_parquet(SPLITS/'test.parquet').sort_values(['user_id','timestamp']).reset_index(drop=True)

required_columns={'user_id','video_id','timestamp','duration_seconds','watched_seconds',
                  'playback_seconds','completion_ratio','segment_count','engagement_weight'}
for name,frame in [('train',train_df),('valid',valid_df),('test',test_df)]:
    absent=required_columns-set(frame.columns)
    if absent: raise ValueError(f'{name} is missing columns: {sorted(absent)}')
    frame['user_id']=frame.user_id.astype(str); frame['video_id']=frame.video_id.astype(str)
    frame['timestamp']=pd.to_numeric(frame.timestamp,errors='coerce').fillna(0).astype('int64')

item_ids=sorted(train_df.video_id.unique()); user_ids=sorted(set(train_df.user_id)|set(valid_df.user_id)|set(test_df.user_id))
item2idx={v:i+1 for i,v in enumerate(item_ids)}; idx2item={i:v for v,i in item2idx.items()}
user2idx={u:i for i,u in enumerate(user_ids)}; idx2user={i:u for u,i in user2idx.items()}
num_items,num_users=len(item2idx),len(user2idx)

def map_frame(frame):
    x=frame[frame.video_id.isin(item2idx)].copy().reset_index(drop=True)
    x['u']=x.user_id.map(user2idx).astype('int64'); x['i']=x.video_id.map(item2idx).astype('int64')
    return x

train=map_frame(train_df);valid=map_frame(valid_df);test=map_frame(test_df)
print({'train':len(train),'validation':len(valid),'test':len(test),'users':num_users,'videos':num_items})
display(pd.DataFrame([
    ['train',len(train),train.u.nunique(),train.i.nunique()],
    ['validation',len(valid),valid.u.nunique(),valid.i.nunique()],
    ['test',len(test),test.u.nunique(),test.i.nunique()]],
    columns=['split','interactions','users','videos']))

## 3. Leakage and chronology audit

These checks must pass before training. A validation or test target may exist in another learner's training data, but it must not appear in the same learner's input history.

In [ ]:
train_max=train.groupby('u').timestamp.max()
valid_by_user=valid.set_index('u'); test_by_user=test.set_index('u')
common_users=sorted(set(train_max.index)&set(valid_by_user.index)&set(test_by_user.index))

checks={
 'train_before_or_at_validation':bool((train_max.loc[common_users].values<=valid_by_user.loc[common_users].timestamp.values).all()),
 'validation_before_or_at_test':bool((valid_by_user.loc[common_users].timestamp.values<=test_by_user.loc[common_users].timestamp.values).all()),
 'validation_items_in_training_catalog':bool(valid.i.isin(set(train.i)).all()),
 'test_items_in_training_catalog':bool(test.i.isin(set(train.i)).all()),
 'same_validation_and_test_users':set(valid.u)==set(test.u),
}
display(pd.DataFrame(checks.items(),columns=['check','passed']))
if not all(checks.values()): raise AssertionError('Leakage/chronology audit failed.')

## 4. Fit behaviour normalization on training data only

Seconds, durations, and segment counts receive `log1p` before standardization. Means and standard deviations are learned only from training interactions and then applied unchanged to validation and test.

In [ ]:
BEHAVIOUR_COLUMNS=['watched_seconds','playback_seconds','duration_seconds',
                   'completion_ratio','segment_count','engagement_weight']
LOG_BEHAVIOUR={'watched_seconds','playback_seconds','duration_seconds','segment_count'}

def raw_behaviour(frame):
    x=frame[BEHAVIOUR_COLUMNS].astype('float32').replace([np.inf,-np.inf],np.nan).fillna(0).copy()
    for c in LOG_BEHAVIOUR:x[c]=np.log1p(x[c].clip(lower=0))
    return x

train_beh_raw=raw_behaviour(train);beh_mean=train_beh_raw.mean();beh_std=train_beh_raw.std().replace(0,1).fillna(1)
def normalized_behaviour(frame):return ((raw_behaviour(frame)-beh_mean)/beh_std).astype('float32').to_numpy()
train_beh=normalized_behaviour(train);valid_beh=normalized_behaviour(valid);test_beh=normalized_behaviour(test)
normalization={'columns':BEHAVIOUR_COLUMNS,'mean':beh_mean.to_dict(),'std':beh_std.to_dict(),'log1p_columns':sorted(LOG_BEHAVIOUR)}
json.dump(normalization,open(REPORTS/'behaviour_normalization.json','w'),indent=2)
display(pd.DataFrame({'mean':beh_mean,'std':beh_std}))

## 5. Build video concept, course, and metadata inputs

Video index 0 is padding. Concept and course index 0 mean unavailable. Metadata normalization also uses the training video catalog only.

In [ ]:
video_index=pd.read_parquet(GRAPH/'video_index.parquet');video_index['video_id']=video_index.video_id.astype(str);video_index['ccid']=video_index.ccid.astype(str)
video_to_ccid=dict(zip(video_index.video_id,video_index.ccid));ccid_to_item={video_to_ccid[v]:item2idx[v] for v in item2idx if v in video_to_ccid}

cv=pd.read_parquet(GRAPH/'concept_video_edges.parquet');cv['concept_id']=cv.concept_id.astype(str);cv['ccid']=cv.ccid.astype(str)
cv=cv[cv.ccid.isin(ccid_to_item)].drop_duplicates(['ccid','concept_id'])
concept_ids=sorted(cv.concept_id.unique());concept2idx={c:i+1 for i,c in enumerate(concept_ids)};idx2concept={i:c for c,i in concept2idx.items()}
item_concepts=np.zeros((num_items+1,CFG.max_concepts_per_video),dtype=np.int64)
for ccid,g in cv.groupby('ccid'):
    ids=[concept2idx[c] for c in g.concept_id.iloc[:CFG.max_concepts_per_video]]
    item_concepts[ccid_to_item[ccid],:len(ids)]=ids

course=pd.read_parquet(GRAPH/'course_video_edges.parquet');course['video_id']=course.video_id.astype(str);course['course_id']=course.course_id.astype(str)
course=course[course.video_id.isin(item2idx)].drop_duplicates('video_id')
course_ids=sorted(course.course_id.unique());course2idx={c:i+1 for i,c in enumerate(course_ids)}
item_course=np.zeros(num_items+1,dtype=np.int64)
for row in course.itertuples():item_course[item2idx[row.video_id]]=course2idx[row.course_id]

metadata=pd.read_parquet(GRAPH/'video_metadata.parquet');metadata['video_id']=metadata.video_id.astype(str)
META_COLUMNS=['duration_seconds','subtitle_sentences','subtitle_characters','concept_count']
for c in META_COLUMNS:
    if c not in metadata:metadata[c]=0
metadata=metadata.drop_duplicates('video_id').set_index('video_id')
meta_table=pd.DataFrame(index=item_ids,columns=META_COLUMNS,dtype='float32')
for c in META_COLUMNS:meta_table[c]=pd.to_numeric(metadata.reindex(item_ids)[c],errors='coerce').fillna(0).astype('float32')
for c in META_COLUMNS:meta_table[c]=np.log1p(meta_table[c].clip(lower=0))
meta_mean=meta_table.mean();meta_std=meta_table.std().replace(0,1).fillna(1)
meta_table=((meta_table-meta_mean)/meta_std).astype('float32')
item_metadata=np.zeros((num_items+1,len(META_COLUMNS)),dtype=np.float32);item_metadata[1:]=meta_table.to_numpy()

print({'concepts':len(concept_ids),'concept_video_edges':len(cv),'courses':len(course_ids),
       'videos_with_concepts':int((item_concepts!=0).any(1).sum()),'videos_with_courses':int((item_course!=0).sum())})
json.dump({'meta_columns':META_COLUMNS,'mean':meta_mean.to_dict(),'std':meta_std.to_dict()},open(REPORTS/'metadata_normalization.json','w'),indent=2)

## 6. Construct chronological histories

Validation history contains training events. Test history contains training events followed by the validation interaction. Negative sampling excludes every known train, validation, and test positive for the same learner.

In [ ]:
def build_history(frame,features):
    out={}
    for u,idx in frame.groupby('u',sort=False).groups.items():
        rows=np.asarray(list(idx));g=frame.loc[rows]
        out[int(u)]={'items':g.i.astype(int).tolist(),'times':g.timestamp.astype('int64').tolist(),
                     'behaviour':features[rows].tolist(),'completion':g.completion_ratio.astype(float).tolist()}
    return out

train_h=build_history(train,train_beh);valid_events=build_history(valid,valid_beh);test_events=build_history(test,test_beh)
eval_users=sorted(set(train_h)&set(valid_events)&set(test_events))
valid_hist={u:copy.deepcopy(train_h[u]) for u in eval_users}
test_hist={}
valid_target={u:valid_events[u]['items'][0] for u in eval_users};test_target={u:test_events[u]['items'][0] for u in eval_users}
valid_completion={u:valid_events[u]['completion'][0] for u in eval_users};test_completion={u:test_events[u]['completion'][0] for u in eval_users}
for u in eval_users:
    h=copy.deepcopy(train_h[u])
    for key in ['items','times','behaviour','completion']:h[key].append(valid_events[u][key][0])
    test_hist[u]=h

all_positive={u:set(train_h[u]['items'])|{valid_target[u],test_target[u]} for u in eval_users}
for u in train_h:
    all_positive.setdefault(u,set(train_h[u]['items']))
assert all(valid_target[u] not in valid_hist[u]['items'] for u in eval_users)
assert all(test_target[u] not in test_hist[u]['items'] for u in eval_users)
print('Leakage-free evaluation users:',len(eval_users))

## 7. Training dataset

Every training example contains only the prefix before its positive target. The target's completion ratio is an auxiliary label and is never added to the input token.

In [ ]:
def left_pad(seq,n,pad):
    seq=list(seq)[-n:];return [pad]*(n-len(seq))+seq

class PrefixDataset(Dataset):
    def __init__(self,histories,max_len):
        self.h=histories;self.max_len=max_len
        self.examples=[(u,t) for u,h in histories.items() for t in range(1,len(h['items']))]
    def __len__(self):return len(self.examples)
    def __getitem__(self,index):
        u,t=self.examples[index];h=self.h[u]
        return (torch.tensor(u),torch.tensor(left_pad(h['items'][:t],self.max_len,0)),
                torch.tensor(left_pad(h['times'][:t],self.max_len,0)),
                torch.tensor(left_pad(h['behaviour'][:t],self.max_len,[0.0]*len(BEHAVIOUR_COLUMNS)),dtype=torch.float32),
                torch.tensor(h['items'][t]),torch.tensor(h['completion'][t],dtype=torch.float32))

train_dataset=PrefixDataset(train_h,CFG.max_len)
train_loader=DataLoader(train_dataset,batch_size=CFG.batch_size,shuffle=True,num_workers=CFG.num_workers,
                        pin_memory=True,persistent_workers=CFG.num_workers>0)
print('Training prefix examples:',len(train_dataset),'batches per epoch:',len(train_loader))

def sample_negatives(users,count):
    result=[]
    for u in users.tolist():
        values=[];known=all_positive[int(u)]
        while len(values)<count:
            x=random.randint(1,num_items)
            if x not in known:values.append(x)
        result.append(values)
    return torch.tensor(result,dtype=torch.long)

## 8. BCE-SASRec architecture

One causal Transformer processes unified interaction tokens. Candidate videos contain static video, concept, course, and metadata components. Learner behaviour and time are used only in historical tokens.

In [ ]:
class BCESASRec(nn.Module):
    def __init__(self,num_items,num_concepts,num_courses,item_concepts,item_course,item_metadata,cfg):
        super().__init__();self.cfg=cfg;d=cfg.hidden_dim
        self.item_emb=nn.Embedding(num_items+1,d,padding_idx=0)
        self.concept_emb=nn.Embedding(num_concepts+1,d,padding_idx=0)
        self.course_emb=nn.Embedding(num_courses+1,d,padding_idx=0)
        self.position_emb=nn.Embedding(cfg.max_len,d);self.time_emb=nn.Embedding(cfg.time_buckets,d,padding_idx=0)
        self.behaviour_mlp=nn.Sequential(nn.Linear(len(BEHAVIOUR_COLUMNS),64),nn.GELU(),nn.Dropout(cfg.dropout),nn.Linear(64,d))
        self.metadata_mlp=nn.Sequential(nn.Linear(len(META_COLUMNS),64),nn.GELU(),nn.Linear(64,d))
        self.concept_query=nn.Linear(d,d,bias=False);self.concept_key=nn.Linear(d,d,bias=False)
        self.event_norm=nn.LayerNorm(d);self.candidate_norm=nn.LayerNorm(d);self.dropout=nn.Dropout(cfg.dropout)
        layer=nn.TransformerEncoderLayer(d,cfg.attention_heads,cfg.feedforward_dim,cfg.dropout,
                batch_first=True,norm_first=True,activation='gelu')
        self.transformer=nn.TransformerEncoder(layer,cfg.transformer_layers);self.output_norm=nn.LayerNorm(d)
        self.completion_head=nn.Sequential(nn.Linear(2*d,d),nn.GELU(),nn.Dropout(cfg.dropout),nn.Linear(d,1))
        self.register_buffer('item_concepts',torch.tensor(item_concepts,dtype=torch.long))
        self.register_buffer('item_course',torch.tensor(item_course,dtype=torch.long))
        self.register_buffer('item_metadata',torch.tensor(item_metadata,dtype=torch.float32))
        self.scale=math.sqrt(d)

    def concept_pool(self,item_idx,return_weights=False):
        ids=self.item_concepts[item_idx];c=self.concept_emb(ids);q=self.concept_query(self.item_emb(item_idx)).unsqueeze(-2)
        # Calculate masking and normalization in FP32. In FP16, 1e-8 becomes zero;
        # videos with no concepts would therefore divide 0 by 0 and produce NaN.
        logits=((q*self.concept_key(c)).sum(-1)/self.scale).float();mask=ids.eq(0)
        logits=logits.masked_fill(mask,-1e9);weights=torch.softmax(logits,dim=-1)
        weights=weights.masked_fill(mask,0.0)
        weights=weights/weights.sum(-1,keepdim=True).clamp_min(1.0)
        pooled=(weights.unsqueeze(-1)*c.float()).sum(-2).to(c.dtype)
        return (pooled,weights,ids) if return_weights else pooled

    def candidate(self,item_idx):
        z=self.item_emb(item_idx)+self.concept_pool(item_idx)+self.course_emb(self.item_course[item_idx])+self.metadata_mlp(self.item_metadata[item_idx])
        return self.candidate_norm(z)

    def time_bucket(self,times):
        gap=torch.zeros_like(times);valid=(times[:,1:]>0)&(times[:,:-1]>0)
        delta=(times[:,1:]-times[:,:-1]).clamp_min(0)
        gap[:,1:]=torch.where(valid,delta,torch.zeros_like(delta))
        bucket=torch.floor(torch.log2(gap.float()+1)).long()+1
        return bucket.clamp(0,self.cfg.time_buckets-1).masked_fill(times.eq(0),0)

    def encode(self,seq,times,behaviour):
        pos=torch.arange(self.cfg.max_len,device=seq.device).unsqueeze(0)
        static=self.candidate(seq);x=static+self.behaviour_mlp(behaviour)+self.time_emb(self.time_bucket(times))+self.position_emb(pos)
        x=self.dropout(self.event_norm(x));causal=torch.triu(torch.ones(self.cfg.max_len,self.cfg.max_len,device=seq.device,dtype=torch.bool),1)
        x=self.transformer(x,mask=causal,src_key_padding_mask=seq.eq(0));return self.output_norm(x[:,-1])

    def sampled_logits(self,h,candidates):return (h.unsqueeze(1)*self.candidate(candidates)).sum(-1)/self.scale
    def all_candidate_embeddings(self):return self.candidate(torch.arange(1,self.item_emb.num_embeddings,device=self.item_emb.weight.device))
    def completion(self,h,item):return torch.sigmoid(self.completion_head(torch.cat([h,self.candidate(item)],-1))).squeeze(-1)

model=BCESASRec(num_items,len(concept_ids),len(course_ids),item_concepts,item_course,item_metadata,CFG).to(device)
print(model)
print('Trainable parameters:',sum(p.numel() for p in model.parameters() if p.requires_grad))

## 9. Correct recommendation metrics

There is one relevant next video for each learner. Consequently, `Precision@K = Recall@K / K`, and the maximum possible F1@10 is 0.1818. NDCG@10 is the primary selection metric.

In [ ]:
def calculate_metrics(ranks,topk_items,item_popularity,num_items,ks=(5,10,20)):
    ranks=np.asarray(ranks,dtype=np.int64);out={'Accuracy@1':float(np.mean(ranks==1)),'MRR':float(np.mean(1/ranks)),
        'MeanRank':float(np.mean(ranks)),'MedianRank':float(np.median(ranks))}
    for k in ks:
        hit=ranks<=k;recall=float(hit.mean());precision=recall/k
        out[f'Precision@{k}']=precision;out[f'Recall@{k}']=recall
        out[f'F1@{k}']=0.0 if recall==0 else float(2*precision*recall/(precision+recall))
        out[f'NDCG@{k}']=float(np.mean(np.where(hit,1/np.log2(ranks+1),0)))
        out[f'MAP@{k}']=float(np.mean(np.where(hit,1/ranks,0)))
    rec=np.asarray(topk_items);out['CatalogCoverage@10']=float(len(np.unique(rec))/num_items)
    total=sum(item_popularity.values());probs=np.array([item_popularity.get(int(i),0.5)/total for i in rec.ravel()])
    out['Novelty@10']=float(np.mean(-np.log2(np.clip(probs,1e-12,None))))
    return out

item_popularity=train.i.value_counts().to_dict()

def tensors_for_users(histories,users):
    seq=torch.tensor([left_pad(histories[u]['items'],CFG.max_len,0) for u in users],device=device)
    times=torch.tensor([left_pad(histories[u]['times'],CFG.max_len,0) for u in users],device=device)
    beh=torch.tensor([left_pad(histories[u]['behaviour'],CFG.max_len,[0.0]*len(BEHAVIOUR_COLUMNS)) for u in users],dtype=torch.float32,device=device)
    return seq,times,beh

@torch.no_grad()
def evaluate(model,histories,targets,target_completion,users,description):
    model.eval();candidate_z=model.all_candidate_embeddings();ranks=[];top_items=[];loss_sum=0.;n=0;completion_errors=[]
    for start in tqdm(range(0,len(users),CFG.eval_batch_size),desc=description,leave=False):
        us=users[start:start+CFG.eval_batch_size];seq,times,beh=tensors_for_users(histories,us)
        h=model.encode(seq,times,beh);scores=(h@candidate_z.T)/model.scale
        if not torch.isfinite(scores).all():
            raise FloatingPointError('Non-finite evaluation scores detected; metrics were not calculated.')
        target=torch.tensor([targets[u]-1 for u in us],device=device)
        for row,u in enumerate(us):
            seen=set(histories[u]['items']);seen.discard(targets[u])
            if seen:scores[row,torch.tensor([i-1 for i in seen],device=device)]=torch.finfo(scores.dtype).min
        loss_sum+=F.cross_entropy(scores,target,reduction='sum').item();n+=len(us)
        target_scores=scores[torch.arange(len(us),device=device),target]
        ranks.extend(((scores>target_scores.unsqueeze(1)).sum(1)+1).cpu().tolist())
        top_items.extend((torch.topk(scores,k=10,dim=1).indices+1).cpu().tolist())
        completion_pred=model.completion(h,target+1).float()
        if not torch.isfinite(completion_pred).all():
            raise FloatingPointError('Non-finite completion predictions detected.')
        completion_pred=completion_pred.cpu().numpy()
        completion_true=np.asarray([target_completion[u] for u in us],dtype=np.float32)
        completion_errors.extend((completion_pred-completion_true).tolist())
    metrics=calculate_metrics(ranks,top_items,item_popularity,num_items,CFG.ks);metrics['Loss']=loss_sum/n
    err=np.asarray(completion_errors);metrics['CompletionMAE']=float(np.mean(np.abs(err)));metrics['CompletionRMSE']=float(np.sqrt(np.mean(err**2)))
    concept_recalls=[]
    for u,recs in zip(users,top_items):
        true=set(item_concepts[targets[u]])-{0};pred=set(item_concepts[np.asarray(recs)].ravel())-{0}
        if true:concept_recalls.append(len(true&pred)/len(true))
    metrics['ConceptRecall@10']=float(np.mean(concept_recalls)) if concept_recalls else float('nan')
    return metrics,ranks,top_items

## 10. Multi-task training with early stopping

Every epoch reports total training loss, its three components, validation loss, validation ranking metrics, elapsed time, and learning rate. The best checkpoint is selected only by validation NDCG@10. Test data is not accessed here.

In [ ]:
def concept_pair_loss(model,h,pos_items):
    ids=model.item_concepts[pos_items];mask=ids.ne(0);has=mask.any(1)
    if not has.any():return h.sum()*0
    first=mask.float().argmax(1);positive=ids[torch.arange(len(ids),device=device),first]
    negative=torch.randint(1,model.concept_emb.num_embeddings,(len(ids),),device=device)
    negative=torch.where(negative.eq(positive),(negative%(model.concept_emb.num_embeddings-1))+1,negative)
    ps=(h*model.concept_emb(positive)).sum(-1)/model.scale;ns=(h*model.concept_emb(negative)).sum(-1)/model.scale
    return -F.logsigmoid(ps[has]-ns[has]).mean()

optimizer=torch.optim.AdamW(model.parameters(),lr=CFG.learning_rate,weight_decay=CFG.weight_decay)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='max',factor=.5,patience=2,min_lr=1e-5)
amp_enabled=CFG.use_mixed_precision and device.type=='cuda'
scaler=torch.amp.GradScaler('cuda',enabled=amp_enabled)
best_score=-float('inf');bad_epochs=0;history=[];checkpoint=CHECKPOINTS/'BCE_SASRec_best.pt'

for epoch in range(1,CFG.max_epochs+1):
    started=time.time();model.train();sums=defaultdict(float);examples=0
    for users,seq,times,behaviour,pos,completion in tqdm(train_loader,desc=f'Epoch {epoch:02d}/{CFG.max_epochs}',leave=False):
        users,seq,times,behaviour,pos,completion=[x.to(device,non_blocking=True) for x in [users,seq,times,behaviour,pos,completion]]
        negatives=sample_negatives(users.cpu(),CFG.negatives).to(device,non_blocking=True)
        candidates=torch.cat([pos.unsqueeze(1),negatives],1);optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type,enabled=amp_enabled):
            h=model.encode(seq,times,behaviour);logits=model.sampled_logits(h,candidates)
            rank_loss=F.cross_entropy(logits,torch.zeros(len(seq),dtype=torch.long,device=device))
            c_loss=concept_pair_loss(model,h,pos);completion_pred=model.completion(h,pos)
            completion_loss=F.mse_loss(completion_pred,completion.clamp(0,1))
            total=rank_loss+CFG.concept_loss_weight*c_loss+CFG.completion_loss_weight*completion_loss
        if not torch.isfinite(total):
            raise FloatingPointError(f'Non-finite training loss in epoch {epoch}. Stop rather than save invalid metrics.')
        scaler.scale(total).backward();scaler.unscale_(optimizer);grad_norm=nn.utils.clip_grad_norm_(model.parameters(),CFG.gradient_clip)
        if not torch.isfinite(grad_norm):
            raise FloatingPointError(f'Non-finite gradient norm in epoch {epoch}.')
        scaler.step(optimizer);scaler.update();bs=len(seq);examples+=bs
        for key,value in [('TrainLoss',total),('RankingLoss',rank_loss),('ConceptLoss',c_loss),('CompletionLoss',completion_loss)]:sums[key]+=float(value.detach())*bs
    val,_,_=evaluate(model,valid_hist,valid_target,valid_completion,eval_users,'Validation')
    scheduler.step(val['NDCG@10'])
    row={'Epoch':epoch,**{k:v/examples for k,v in sums.items()},
         **{f'Val_{k}':v for k,v in val.items()},'LearningRate':optimizer.param_groups[0]['lr'],'Seconds':time.time()-started}
    history.append(row);pd.DataFrame(history).to_csv(REPORTS/'epoch_history.csv',index=False)
    print(f"Epoch {epoch:02d}: train={row['TrainLoss']:.4f} valid={row['Val_Loss']:.4f} "
          f"val Recall@10={row['Val_Recall@10']:.4f} val NDCG@10={row['Val_NDCG@10']:.4f} "
          f"val MRR={row['Val_MRR']:.4f} lr={row['LearningRate']:.2e} time={row['Seconds']:.1f}s")
    improved=val['NDCG@10']>best_score+CFG.early_stopping_min_delta
    if improved:
        best_score=val['NDCG@10'];bad_epochs=0
        torch.save({'model_state':model.state_dict(),'epoch':epoch,'validation_metrics':val,'config':asdict(CFG)},checkpoint)
        print('  Saved new best checkpoint.')
    else:bad_epochs+=1
    if epoch>=CFG.minimum_epochs and bad_epochs>=CFG.early_stopping_patience:
        print(f'Early stopping after epoch {epoch}; best validation NDCG@10={best_score:.4f}.');break

print('Epochs completed:',len(history),'Best checkpoint:',checkpoint)

## 11. Restore the best checkpoint and evaluate the test split once

This cell is the only point where the test targets are scored. This separation prevents test-set model selection.

In [ ]:
saved=torch.load(checkpoint,map_location=device);model.load_state_dict(saved['model_state']);model.eval()
test_metrics,test_ranks,test_top10=evaluate(model,test_hist,test_target,test_completion,eval_users,'Final test')
result={'Model':'BCE-SASRec','BestEpoch':saved['epoch'],'EpochsCompleted':len(history),
        'BestValidationNDCG@10':saved['validation_metrics']['NDCG@10'],**{f'Test_{k}':v for k,v in test_metrics.items()}}
result_df=pd.DataFrame([result]);display(result_df.T.rename(columns={0:'Value'}))
result_df.to_csv(REPORTS/'test_metrics.csv',index=False)

recommendations=[]
for u,items in zip(eval_users,test_top10):
    recommendations.append({'user_id':idx2user[u],'actual_video_id':idx2item[test_target[u]],
                            **{f'rank_{j+1}':idx2item[i] for j,i in enumerate(items)}})
pd.DataFrame(recommendations).to_parquet(REPORTS/'test_top10_recommendations.parquet',index=False)
json.dump({'model':'BCE-SASRec','config':asdict(CFG),'best_epoch':saved['epoch'],
           'validation_metrics':saved['validation_metrics'],'test_metrics':test_metrics,
           'files':{'checkpoint':str(checkpoint),'epoch_history':str(REPORTS/'epoch_history.csv'),
                    'test_metrics':str(REPORTS/'test_metrics.csv')}},open(REPORTS/'experiment_manifest.json','w'),indent=2)
print('Saved all final outputs under:',OUT)

## 12. Plot every epoch and mark the selected checkpoint

In [ ]:
hist=pd.DataFrame(history);best_epoch=int(saved['epoch'])
fig,axes=plt.subplots(1,3,figsize=(17,4.5))
axes[0].plot(hist.Epoch,hist.TrainLoss,label='Train multi-task loss');axes[0].plot(hist.Epoch,hist.Val_Loss,label='Validation full-catalog loss')
axes[0].set_title('Loss by epoch');axes[0].legend()
axes[1].plot(hist.Epoch,hist['Val_Recall@10'],label='Recall@10');axes[1].plot(hist.Epoch,hist['Val_NDCG@10'],label='NDCG@10');axes[1].plot(hist.Epoch,hist.Val_MRR,label='MRR')
axes[1].set_title('Validation ranking metrics');axes[1].legend()
axes[2].plot(hist.Epoch,hist.RankingLoss,label='Ranking');axes[2].plot(hist.Epoch,hist.ConceptLoss,label='Concept');axes[2].plot(hist.Epoch,hist.CompletionLoss,label='Completion')
axes[2].set_title('Training loss components');axes[2].legend()
for ax in axes:ax.axvline(best_epoch,color='red',linestyle='--',alpha=.7,label='Best epoch');ax.set_xlabel('Epoch');ax.grid(alpha=.25)
fig.tight_layout();fig.savefig(REPORTS/'training_curves.png',dpi=180,bbox_inches='tight');plt.show()

## 13. Counterfactual recommendation explanation

The explanation masks each historical interaction and measures how much the selected recommendation score decreases. It also reports the candidate video's learned concept-attention weights.

In [ ]:
@torch.no_grad()
def explain_recommendation(user_index,recommended_item,top_history=5,top_concepts=8):
    model.eval();hdata=test_hist[user_index];seq,times,beh=tensors_for_users(test_hist,[user_index])
    target=torch.tensor([recommended_item],device=device);base_h=model.encode(seq,times,beh)
    base_score=float((base_h*model.candidate(target)).sum()/model.scale)
    if not math.isfinite(base_score):raise FloatingPointError('Explanation score is non-finite; checkpoint is invalid.')
    evidence=[]
    valid_positions=torch.where(seq[0].ne(0))[0]
    for p in valid_positions:
        masked_seq=seq.clone();masked_times=times.clone();masked_beh=beh.clone()
        removed=int(masked_seq[0,p]);masked_seq[0,p]=0;masked_times[0,p]=0;masked_beh[0,p]=0
        score=float((model.encode(masked_seq,masked_times,masked_beh)*model.candidate(target)).sum()/model.scale)
        evidence.append({'video_id':idx2item[removed],'score_drop':base_score-score})
    evidence=sorted(evidence,key=lambda x:x['score_drop'],reverse=True)[:top_history]
    _,weights,ids=model.concept_pool(target,return_weights=True)
    concepts=[]
    for cid,w in zip(ids[0].tolist(),weights[0].tolist()):
        if cid:concepts.append({'concept_id':idx2concept[cid],'attention_weight':w})
    concepts=sorted(concepts,key=lambda x:x['attention_weight'],reverse=True)[:top_concepts]
    return {'user_id':idx2user[user_index],'recommended_video_id':idx2item[recommended_item],
            'base_score':base_score,'important_history':evidence,'important_concepts':concepts}

sample_user=eval_users[0];sample_item=test_top10[0][0]
explanation=explain_recommendation(sample_user,sample_item)
print(json.dumps(explanation,indent=2,ensure_ascii=False))
json.dump(explanation,open(EXPLANATIONS/'sample_explanation.json','w'),indent=2,ensure_ascii=False)

## Output files

| File | Purpose |
|---|---|
| `checkpoints/BCE_SASRec_best.pt` | Best validation checkpoint |
| `reports/epoch_history.csv` | Every executed epoch and all validation metrics |
| `reports/test_metrics.csv` | Final untouched test metrics |
| `reports/test_top10_recommendations.parquet` | Top-10 recommendations for each test learner |
| `reports/training_curves.png` | Loss and metric curves |
| `reports/experiment_manifest.json` | Parameters and experiment provenance |
| `explanations/sample_explanation.json` | Counterfactual history and concept evidence |

The main comparison metric is test NDCG@10. Recall@10 and MRR are secondary recommendation metrics. Explanation quality should later be evaluated with fidelity, sparsity, stability, and concept-path correctness.